# FinSight - 01 . Data Exploration

All data is **synthetic** (see `docs/ground_truth/`). This notebook does light EDA: shapes, missing values (raw vs cleaned), distributions and category/time patterns.

It reads files produced by the pipeline; run `python src/run_pipeline.py` first.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
ROOT = Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
raw = ROOT/'data'/'raw'; proc = ROOT/'data'/'processed'

## 1. Load raw and cleaned data

In [ ]:
tables=['customers','merchants','campaigns','transactions','campaign_interactions']
raw_d={t:pd.read_csv(raw/f'{t}.csv') for t in tables}
proc_d={t:pd.read_csv(proc/f'{t}.csv') for t in tables}
for t in tables:
    print(f'{t:24s} raw={len(raw_d[t]):>6,}  cleaned={len(proc_d[t]):>6,}')

## 2. Data quality: missing values before cleaning
Note: `campaign_id` is legitimately null for non-campaign transactions, so it is not a defect.

In [ ]:
miss = raw_d['transactions'].isna().sum()
print(miss[miss>0])
print('\nmerchant categories (raw) - note the malformed ones:')
print(raw_d['merchants']['category'].value_counts().tail(8))

## 3. Transaction amount distribution
Right-skewed (log-normal). The far-right tail includes the deliberately injected extreme values.

In [ ]:
amt = proc_d['transactions']['transaction_amount']
fig,ax=plt.subplots(1,2,figsize=(12,4))
sns.histplot(amt[amt<amt.quantile(0.99)],bins=50,ax=ax[0]); ax[0].set_title('Amount (<99th pct)')
sns.histplot(np.log10(amt[amt>0]),bins=50,ax=ax[1]); ax[1].set_title('log10(amount)')
plt.tight_layout()

## 4. Revenue and volume by category

In [ ]:
t=proc_d['transactions'].merge(proc_d['merchants'][['merchant_id','category']],on='merchant_id',how='left')
cat=t.groupby('category')['transaction_amount'].agg(['sum','count']).sort_values('sum',ascending=False)
cat.columns=['revenue','transactions']; display(cat)
cat['revenue'].plot(kind='bar',figsize=(9,4),title='Revenue by category'); plt.tight_layout()

## 5. Monthly transaction volume (seasonality)
Travel peaks in vacation months (see B5 in the ground truth).

In [ ]:
t['month']=pd.to_datetime(t['transaction_date']).dt.strftime('%Y-%m')
overall=t.groupby('month').size()
travel=t[t['category']=='Travel'].groupby('month').size()
fig,ax=plt.subplots(figsize=(10,4))
overall.plot(ax=ax,label='All categories')
(travel*(overall.mean()/travel.mean())).plot(ax=ax,label='Travel (scaled)')
ax.legend(); ax.set_title('Monthly transaction volume'); plt.tight_layout()

### Takeaways
- Cleaning removes duplicates, orphans, negatives and bad dates; extreme values are kept but flagged.
- Amounts are log-normal - relevant later for anomaly detection (IQR in log space).
- Category revenue and Travel seasonality match the documented behavioural design.